# Real-Cell Spatial Walk

A different, deliberately simpler take on "dynamics" than the earlier
attempts: instead of fitting a vector field or a space field and querying
arbitrary points, this walk **never leaves real cells**. Every point on
the resulting path is an actual measured cell -- nothing here is decoded
from an interpolated or extrapolated coordinate, so the off-manifold
drift that broke every earlier dynamics attempt structurally can't happen
here.

**Algorithm, per step:**
1. Bin pseudotime (equal-count bins, as in the earlier notebooks).
2. From the current real cell, restrict candidates to its **k nearest
   real cells** within the *next* bin -- measured directly in
   standardized `(pseudotime, UMAP1, UMAP2)` space, the same space being
   plotted, not the 10D scVI latent space.
3. Move to whichever of those k candidates is closest.
4. Repeat, one real cell per bin, until the end of pseudotime.

**Why 3D space instead of latent space:** matching in the 10D latent
space picks the most *transcriptionally* similar next cell, but UMAP
doesn't preserve latent-space distance -- two cells close in latent space
can land far apart in the 2D projection, and vice versa. That mismatch is
what produced visually incoherent, jumpy paths. Operating directly in the
plotted space trades some biological precision for a path that's visually
coherent by construction.

Also included: an actual-vs-decoded expression plot. Every cell on this
path has REAL measured expression already known, so decoding its own
latent vector through the frozen scVI decoder and comparing is a direct
fidelity check on the decoder itself.

**Prerequisite:** loads `artifacts/adata_filtered.h5ad` and
`artifacts/scvi_model/` from the earlier pipeline notebook.


## 0. Config

In [ ]:
import os

ARTIFACT_DIR = "artifacts"
SCVI_MODEL_DIR = os.path.join(ARTIFACT_DIR, "scvi_model")
FILTERED_ADATA_PATH = os.path.join(ARTIFACT_DIR, "adata_filtered.h5ad")

PSEUDOTIME_KEY = "Pseudotime"
BATCH_KEY = "replicate_id"

N_BINS = 500           # equal-count pseudotime bins -- finer waypoint spacing, still real cells only
K_NEIGHBORS = 15       # candidates considered per step, before correlation picks the winner
RANDOM_SEED = 0

GENES_OF_INTEREST = ["Sox2", "Olig2", "Nkx6-1", "Pax6", "Nkx2-2", "Foxa2"]


## 1. Load the filtered AnnData and the scVI model

In [ ]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import torch

assert os.path.exists(FILTERED_ADATA_PATH), f"{FILTERED_ADATA_PATH} not found -- run the earlier pipeline notebook first."
assert os.path.exists(SCVI_MODEL_DIR), f"{SCVI_MODEL_DIR} not found -- run the earlier pipeline notebook first."

adata_filtered = sc.read_h5ad(FILTERED_ADATA_PATH)
print(adata_filtered)

loaded_scvi = scvi.model.SCVI.load(SCVI_MODEL_DIR)
loaded_scvi.module.eval()
adata_ref = loaded_scvi.adata
print("scVI model loaded.")


## 2. Bin pseudotime

Equal-count bins, same reasoning as earlier notebooks: this dataset's
pseudotime is density-skewed (dense progenitor pool early, sparse branch
tips late), so equal-count keeps every bin statistically usable.


In [ ]:
n_cells = len(adata_filtered)
n_unique_pt = adata_filtered.obs[PSEUDOTIME_KEY].nunique()
print(f"Total cells: {n_cells}")
print(f"Average cells per bin at N_BINS={N_BINS}: {n_cells/N_BINS:.1f}  (want this comfortably > K_NEIGHBORS={K_NEIGHBORS})")
print(f"Unique pseudotime values: {n_unique_pt}  (want this comfortably > N_BINS, or qcut will collapse some bins)")

# duplicates='drop' is a safety net: if there aren't enough unique values to
# carve N_BINS distinct edges, this silently gives fewer bins instead of
# crashing -- the right failure mode here, not a hard stop.
adata_filtered.obs["pt_bin"] = pd.qcut(adata_filtered.obs[PSEUDOTIME_KEY], q=N_BINS, labels=False, duplicates="drop")
actual_n_bins = adata_filtered.obs["pt_bin"].nunique()
if actual_n_bins < N_BINS:
    print(f"Note: requested {N_BINS} bins, got {actual_n_bins} after dropping duplicate edges.")
print(adata_filtered.obs["pt_bin"].value_counts().sort_index().describe())


## 3. The walk: kNN in (pseudotime, UMAP1, UMAP2) space

Restricting candidates and picking the next step **directly in the space
being plotted** — not the 10D scVI latent space. This trades some
biological precision (two cells near each other in UMAP aren't
guaranteed to be the most transcriptionally similar real next state,
since UMAP distorts distances for visualization) for a path that's
visually coherent by construction, which is what we actually want here.

Each axis is standardized (z-scored) before computing distance — `t`
ranges roughly 0-95 while UMAP coordinates are roughly -10 to 15, and
without normalizing, whichever axis happens to have the larger raw scale
would dominate the distance calculation for no principled reason.


In [ ]:
t_all = adata_filtered.obs[PSEUDOTIME_KEY].values
u_all = adata_filtered.obsm["X_umap"]

t_mean, t_std = t_all.mean(), t_all.std()
u_mean, u_std = u_all.mean(axis=0), u_all.std(axis=0)

coords_std = np.column_stack([
    (t_all - t_mean) / t_std,
    (u_all[:, 0] - u_mean[0]) / u_std[0],
    (u_all[:, 1] - u_mean[1]) / u_std[1],
])


def real_cell_spatial_walk(start_idx, k_neighbors=K_NEIGHBORS):
    """One path: at each step, restrict to the k nearest real cells (in
    standardized t/UMAP1/UMAP2 space) within the next bin, then move to
    whichever one is closest -- nearest in the literal space being plotted,
    nothing more abstract than that."""
    bin_labels = adata_filtered.obs["pt_bin"].values
    path_idx = [start_idx]
    current_idx = start_idx
    current_bin = bin_labels[start_idx]

    for next_bin in range(current_bin + 1, actual_n_bins):
        candidate_idx = np.where(bin_labels == next_bin)[0]
        if len(candidate_idx) == 0:
            continue

        c_current = coords_std[current_idx]
        c_candidates = coords_std[candidate_idx]
        dists = np.linalg.norm(c_candidates - c_current, axis=1)
        k = min(k_neighbors, len(candidate_idx))
        nn_local = np.argpartition(dists, k - 1)[:k]

        chosen_local = nn_local[np.argmin(dists[nn_local])]
        current_idx = candidate_idx[chosen_local]
        path_idx.append(current_idx)

    return np.array(path_idx)


In [ ]:
# Start from a real cell at low pseudotime (not a centroid -- lesson from
# earlier notebooks: the mean of a curved cloud can land off-manifold even
# though every individual cell on it is real).
early_mask = adata_filtered.obs[PSEUDOTIME_KEY] < 5
start_idx = np.where(early_mask.values)[0][0]

path = real_cell_spatial_walk(start_idx)
print(f"Path length: {len(path)} real cells")


## 4. Decode wiring — and the actual-vs-decoded validation plot

Every cell on this path has real, measured expression already. Decoding
its own latent vector through the frozen scVI decoder and comparing
against that real measurement is a direct fidelity check on the decoder
itself, not on the walk.


In [ ]:
ref_batch_name = adata_ref.obs[BATCH_KEY].value_counts().idxmax()
median_lib_size = (adata_ref.obs["lib_size"].values.copy() if "lib_size" in adata_ref.obs.columns
                    else np.asarray(adata_ref.X.sum(axis=1)).flatten())


def decode_latent_to_expression(z_batch, batch_codes, library_sizes):
    n = z_batch.shape[0]
    z_t = torch.tensor(z_batch, dtype=torch.float32)
    library_log = torch.tensor(np.log(library_sizes), dtype=torch.float32).view(n, 1)
    batch_idx = torch.tensor(batch_codes, dtype=torch.long).view(n, 1)
    with torch.no_grad():
        gen_out = loaded_scvi.module.generative(z=z_t, library=library_log, batch_index=batch_idx)
    px = gen_out["px"]
    return px.mu.detach().cpu().numpy() if hasattr(px, "mu") else px.mean.detach().cpu().numpy()


In [ ]:
# Decode every cell ON THE PATH using its own real z, real batch, real library size
path_z = adata_filtered.obsm["X_scVI"][path]
path_batch_codes = adata_filtered.obs[BATCH_KEY].astype("category").cat.codes.values[path]
path_lib = (adata_filtered.obs["lib_size"].values[path] if "lib_size" in adata_filtered.obs.columns
            else np.asarray(adata_filtered.X[path].sum(axis=1)).flatten())

decoded_expr = decode_latent_to_expression(path_z, path_batch_codes, path_lib)
actual_expr = (np.asarray(adata_filtered.X[path].todense())
               if hasattr(adata_filtered.X, "todense") else adata_filtered.X[path])

per_cell_corr = [np.corrcoef(decoded_expr[i], actual_expr[i])[0, 1] for i in range(len(path))]
print(f"Per-cell decoded-vs-actual correlation along the path: mean={np.mean(per_cell_corr):.3f}, "
      f"min={np.min(per_cell_corr):.3f}, max={np.max(per_cell_corr):.3f}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) Actual vs decoded scatter, every gene x every path cell
ax = axes[0]
ax.scatter(actual_expr.flatten(), decoded_expr.flatten(), s=3, alpha=0.15, color="#4fd1c5")
lims = [0, max(actual_expr.max(), decoded_expr.max())]
ax.plot(lims, lims, color="gray", linestyle="--", linewidth=1, label="y = x")
ax.set_xlabel("actual measured expression"); ax.set_ylabel("decoded (reconstructed) expression")
ax.set_title("Decoder fidelity across all genes, all path cells")
ax.legend()

# (b) Per-gene trajectory: actual (solid) vs decoded (dashed)
ax2 = axes[1]
gene_names = adata_filtered.var_names.values
genes_present = [g for g in GENES_OF_INTEREST if g in gene_names]
path_t = adata_filtered.obs[PSEUDOTIME_KEY].values[path]

for g in genes_present:
    gi = list(gene_names).index(g)
    line, = ax2.plot(path_t, actual_expr[:, gi], linewidth=2, label=f"{g} (actual)")
    ax2.plot(path_t, decoded_expr[:, gi], linewidth=2, linestyle="--", color=line.get_color(),
             label=f"{g} (decoded)")
ax2.set_xlabel("pseudotime"); ax2.set_ylabel("expression")
ax2.set_title("Actual vs decoded, along the path")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "real_cell_walk_validation.png"), dpi=150)
plt.show()


**Reading the left plot:** points hugging the `y = x` line mean the
decoder reconstructs real cells faithfully -- this is a property of scVI
itself, validated here on the specific cells this walk actually visited,
not a cherry-picked sample.

**Reading the right plot:** solid and dashed lines for the same gene
should track closely if the decoder preserves the real expression dynamics
along this lineage, not just the overall correlation.


## 5. The path through the landscape

Same 3D view as every previous notebook, for visual continuity --
background real cells, path overlaid.


## 5. The path through the landscape

Using `adata_filtered.obsm["X_umap"]` directly -- the same embedding
already in your data, not a freshly-fit one. No need to fit a new
reducer here: every point on this path is a real cell that already has a
real UMAP coordinate, so there's nothing new to project.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

bg_t = adata_filtered.obs[PSEUDOTIME_KEY].values
bg_u = adata_filtered.obsm["X_umap"]
ax.scatter(bg_t, bg_u[:,0], bg_u[:,1], s=2, alpha=0.10, c=bg_t, cmap="viridis")

path_u = adata_filtered.obsm["X_umap"][path]
ax.plot(path_t, path_u[:,0], path_u[:,1], color="lime", linewidth=2, marker="o", markersize=3)

ax.set_xlabel("pseudotime"); ax.set_ylabel("UMAP1"); ax.set_zlabel("UMAP2")
ax.set_title("Real-cell walk through the landscape (3D-space nearest-neighbor)")
plt.tight_layout()
plt.show()
